In [1]:
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from agent_graph import StagePlayWriter,input_message, StagePlayState
from langgraph.types import Command
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver
import time

from uuid import uuid4


### Setup agent graph

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
# tools = [get_character_description, create_character, human_assistance]

# thread_id = uuid4()
thread_id = "this_thread_2"

graph_config: RunnableConfig = RunnableConfig({"configurable": {"thread_id": thread_id}})

playwriter = StagePlayWriter(
    llm=llm,
    themes= """Loss of innocence, Becoming Psychologically whole, Jungian Psychology, Bildung""",
    vibe= """Subtly, Weird and funky""",
    setting= "Tam Tamoree, fictional town in German Bavaria",
    number_of_chapters= 4
)
conn_checkptr = "db/graph_checkpoints/checkpoints.db"


### Graph invoke and continue functions

In [3]:
async def start_graph(init_message: str ,graph_config:  RunnableConfig) -> dict[str, str]:
    """Start the agent application
    """
    async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        async for event in (playwriter
                            .build_graph(checkpointer)
                            .astream(input=input_message(init_message), config= graph_config, stream_mode="values")):

            context = event["context"][-1]
            if isinstance(context, tuple):
                print(event)
            else:
                context.pretty_print()

    return {"status": "Graph started, may be paused"}


async def resume_graph(input: str, graph_config) -> dict[str, str]:
    """Resume graph after break from human in the loop tool call
    """
    resume_input = Command(resume= {"data": input}) 
    async with AsyncSqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        async for event in (playwriter
                            .build_graph(checkpointer)
                            .astream(input=resume_input, config= graph_config, stream_mode="values")):

            context = event["context"][-1]
            if isinstance(context, tuple):
                print(event)
            else:
                context.pretty_print()
    return {"status": "Resumed, may stopp again"}



In [4]:
def run_graph(message: str, graph_config:  RunnableConfig) -> dict[str, str]:
    """Start the agent application
    """
    with SqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
        for event in (playwriter
                        .build_graph(checkpointer)
                        .stream(input=input_message(message), # type: ignore
                                config= graph_config, 
                                stream_mode="values")): 

            context = event["context"][-1] 
            print(context)
    return {"status": "Graph started, may be paused"}


In [ ]:
with SqliteSaver.from_conn_string(conn_checkptr) as checkpointer:
    graph = playwriter.build_graph(checkpointer)

    for event in graph.stream(input=input_message("""Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        """), 
        config= {"configurable": {"thread_id": "some_thread_this_is"}}, 
        stream_mode="values"):
        print(event["context"][-1]) 

### Call graph 

In [18]:
await start_graph(init_message= """Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        """, graph_config= graph_config)

================================ Human Message =================================

Narrator: It is a sunny wistful day in Tam Tamouree.
        Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
        Luna:
        
================================== Ai Message ==================================
Tool Calls:
  create_character (call_jZbH4k9FgopMEMsyXrpTQPha)
 Call ID: call_jZbH4k9FgopMEMsyXrpTQPha
  Args:
    character_name: Anya
    age: 50
    gender: F
    disposition: Mysterious
    relationships_in: {'Townspeople': 'Eccentric local', 'Luna': 'Guiding mentor'}
  create_character (call_p7LIYZsBmMX3yRGXEYbiV773)
 Call ID: call_p7LIYZsBmMX3yRGXEYbiV773
  Args:
    character_name: Max
    age: 19
    gender: M
    disposition: Reckless
    relationships_in: {'Townspeople': 'Aloof', 'Rory': 'Frenemy'}
  create_character (call_X9STEDPuP2CcngWBGNzYnHhY)
 Call ID: call_X9STEDPuP2CcngWBGNzYnHhY
  Args:
    character_name: Franzi
    ag

{'status': 'Graph started, may be paused'}

In [14]:
await resume_graph("""Luna:
                   Oh my god! My blood is ants. I am dying Swedenborg. Oh! The pain!! I can't see. The creepy crawlies, they're within me. Help! """, 
                   graph_config= graph_config)

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_3sdylspYBQ92rZlu1JZ7ROjh)
 Call ID: call_3sdylspYBQ92rZlu1JZ7ROjh
  Args:
    query: Please provide the next line for Captain Flask after the following context: 

Luna: Why does everyone act like they’re so afraid of Rory? He’s just a guy, right? 

Captain Flask: But Luna, sometimes a guy like Rory can be a storm you can’t escape from.
================================= Tool Message =================================
Name: human_assistance

Luna:
                   Oh my god! My blood is ants. I am dying Swedenborg. Oh! The pain!! I can't see. The creepy crawlies, they're within me. Help! 
================================== Ai Message ==================================

It seems we have reached the end of the current chapter with 15 lines already set. As we prepare to transition into the next chapter of the play, I suggest we can introduce a pivotal moment that propels th

{'status': 'Resumed, may stopp again'}

In [16]:
await resume_graph("""Swedenborg:
                   Whatever! You're being weird today """, graph_config)

================================ Human Message =================================

Narrator: 


{'status': 'Resumed, may stopp again'}

In [3]:
checkptr = InMemorySaver()
graph = playwriter.build_graph(checkptr)
graph_config = RunnableConfig({"configurable": {"thread_id": "some_thread_this_is"}})

for event in graph.stream(input=input_message("""Narrator: It is a sunny wistful day in Tam Tamouree.
    Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
    Luna:
    """), 
    config= graph_config, 
    stream_mode="values"):
 
    print(event["context"][-1].content) 

Narrator: It is a sunny wistful day in Tam Tamouree.
    Swedenborg and Luna lazily scout over the townspeople from their hidden vantage point atop the old church.
    Luna:
    
content='' additional_kwargs={'tool_calls': [{'id': 'call_Um8I4uWCEkAncbYiTagyxjEv', 'function': {'arguments': '{"query":"What would you like Luna to say next?"}', 'name': 'human_assistance'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 1186, 'total_tokens': 1210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1152}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-C88Hk8CMyVbMsj6qw1x4LrqlufjjA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--6a21ac1e-859b-4957-9ec2-083e17e8fbdc-0' tool_calls=[{'name': 'h